# **TRABAJO DE IA GENERATIVA**
> Se usa la información hostórica de SBS

> Descargamos desde portal de información

In [1]:
import os
import pandas as pd
from langchain_core.documents import Document

c:\proyecto_IA_V2\proyecto_final_ia\env_rag\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
current_dir = os.getcwd() 

# Esto sube un nivel a la carpeta 'RAG_IA' y entra a 'data'
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'Output_data_csv', 'documents')
DATA_PATH

'c:\\proyecto_IA_V2\\proyecto_final_ia\\Output_data_csv\\documents'

In [3]:
import os
import runpy

# Subir un nivel desde el directorio actual y obtener la ruta del directorio principal del proyecto
project_dir = os.path.join(os.path.dirname(os.getcwd()), 'Scripts')

# Ruta completa al archivo Extrae.py dentro de la carpeta Scripts
module_path = os.path.join(project_dir, 'Extrae.py')

# Verificar si el archivo Extrae.py existe en la ruta
if os.path.exists(module_path):
    print(f"Ejecutando {module_path}...")
    # Ejecutar el archivo Extrae.py directamente
    module_globals = runpy.run_path(module_path)  # Ejecutar el archivo Extrae.py y almacenar los objetos del módulo

    # Obtener la función 'descargar_datos' desde el espacio de nombres global
    descargar_datos = module_globals.get('descargar_datos')

    if descargar_datos:
        # Obtener la ruta donde se deben guardar los archivos descargados
        ruta_descarga = os.path.join(os.path.dirname(os.getcwd()), 'Output_data_csv', 'documents')

        # Llamar a la función con el rango de años (2015 a 2025) y la ruta destino
        descargar_datos(2015, 2025, ruta_destino=ruta_descarga)  # Pasar la ruta de descarga
    else:
        print("La función 'descargar_datos' no se encontró en el archivo Extrae.py.")
else:
    print(f"No se encontró el archivo {module_path}. Verifica que la ruta sea correcta.")

Ejecutando c:\proyecto_IA_V2\proyecto_final_ia\Scripts\Extrae.py...

Resumen de descargas:
Archivos descargados: 132
Archivos no descargados: 0

Archivos descargados:
- 2015-Enero
- 2015-Febrero
- 2015-Marzo
- 2015-Abril
- 2015-Mayo
- 2015-Junio
- 2015-Julio
- 2015-Agosto
- 2015-Setiembre
- 2015-Octubre
- 2015-Noviembre
- 2015-Diciembre
- 2016-Enero
- 2016-Febrero
- 2016-Marzo
- 2016-Abril
- 2016-Mayo
- 2016-Junio
- 2016-Julio
- 2016-Agosto
- 2016-Setiembre
- 2016-Octubre
- 2016-Noviembre
- 2016-Diciembre
- 2017-Enero
- 2017-Febrero
- 2017-Marzo
- 2017-Abril
- 2017-Mayo
- 2017-Junio
- 2017-Julio
- 2017-Agosto
- 2017-Setiembre
- 2017-Octubre
- 2017-Noviembre
- 2017-Diciembre
- 2018-Enero
- 2018-Febrero
- 2018-Marzo
- 2018-Abril
- 2018-Mayo
- 2018-Junio
- 2018-Julio
- 2018-Agosto
- 2018-Setiembre
- 2018-Octubre
- 2018-Noviembre
- 2018-Diciembre
- 2019-Enero
- 2019-Febrero
- 2019-Marzo
- 2019-Abril
- 2019-Mayo
- 2019-Junio
- 2019-Julio
- 2019-Agosto
- 2019-Setiembre
- 2019-Octubre
- 2019-

# **2. Load dataset**

In [4]:

documents = []

archivos_excel = []

for root, dirs, files in os.walk(DATA_PATH):
    for f in files:
        if f.endswith((".xls", ".xlsx")):
            archivos_excel.append(os.path.join(root, f))

print("📂 Excel encontrados:", len(archivos_excel))


for archivo in archivos_excel:

    df = pd.read_excel(
        archivo,
        sheet_name="Ratio de capital CM",
        header=None,
        dtype=str
    )

    df = df.fillna("").astype(str)

    # Buscar fila ENTIDAD
    try:
        fila_header = df[df[0].str.strip() == "ENTIDAD"].index[0]
    except IndexError:
        print("⚠ No se encontró ENTIDAD en:", archivo)
        continue

    # 🔥 Periodo solo YYYY-MM
    periodo_raw = df.iloc[1, 0]
    try:
        periodo = pd.to_datetime(periodo_raw).strftime("%Y-%m")
    except:
        periodo = str(periodo_raw)[:7]

    for i in range(fila_header + 2, len(df)):

        entidad = df.iloc[i, 0].strip()

        if entidad == "":
            continue

        if not entidad.startswith("CMAC") and not entidad.startswith("CMCP"):
            continue

        riesgo_credito = df.iloc[i, 1].strip()
        riesgo_mercado = df.iloc[i, 2].strip()
        riesgo_operacional = df.iloc[i, 3].strip()
        total_requerimiento = df.iloc[i, 4].strip()
        patrimonio_efectivo = df.iloc[i, 5].strip()
        ratio_capital = df.iloc[i, 6].strip()

        texto = (
            f"En el periodo {periodo}, la entidad {entidad} presentó un requerimiento de patrimonio efectivo "
            f"por riesgo de crédito de {riesgo_credito}, por riesgo de mercado de {riesgo_mercado} "
            f"y por riesgo operacional de {riesgo_operacional}, sumando un total requerido de "
            f"{total_requerimiento}. El patrimonio efectivo total fue de {patrimonio_efectivo} "
            f"y el Ratio de Capital Global alcanzó {ratio_capital}."
        )

        documents.append(
            Document(
                page_content=texto,
                metadata={
                    "source": os.path.basename(archivo),
                    "periodo": periodo,
                    "entidad": entidad
                }
            )
        )

print("✅ Total documentos creados:", len(documents))

📂 Excel encontrados: 132
✅ Total documentos creados: 1566


In [5]:
print(documents[0].page_content)

En el periodo 2015-04, la entidad CMAC Arequipa presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 291153.96333, por riesgo de mercado de 9503.08658 y por riesgo operacional de 73304.60677, sumando un total requerido de 373961.65668. El patrimonio efectivo total fue de 540653.3619700001 y el Ratio de Capital Global alcanzó 14.46.


## **3. Chunking + overlap**

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunking académico pero seguro
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # mayor que el tamaño promedio del párrafo
    chunk_overlap=100,   # pequeño solapamiento por continuidad semántica
)

chunks = text_splitter.split_documents(documents)

print("✅ Total chunks creados:", len(chunks))

✅ Total chunks creados: 1566


In [7]:
print(len(documents))
print(len(chunks))

1566
1566


## **4. Embeddings**

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# ==========================================================
# 1️⃣ MODELO
# ==========================================================

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ==========================================================
# 2️⃣ EMBEDDINGS
# ==========================================================

texts = [chunk.page_content for chunk in chunks]

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

print("Total embeddings:", embeddings.shape[0])
print("Dimensión:", embeddings.shape[1])

# ==========================================================
# 3️⃣ FAISS HNSW (AQUÍ VA EL CAMBIO)
# ==========================================================

dimension = embeddings.shape[1]

index = faiss.IndexHNSWFlat(
    dimension,
    32,
    faiss.METRIC_INNER_PRODUCT   # 🔥 IMPORTANTE
)

index.hnsw.efConstruction = 200
index.hnsw.efSearch = 100   # 🔥 mejora precisión

index.add(embeddings)

print("Vectores indexados:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total embeddings: 1566
Dimensión: 384
Vectores indexados: 1566


📌 Celda 4 — Ver ejemplo de chunk

In [9]:
print("Chunks:", len(chunks))
print("Embeddings:", embeddings.shape[0])

Chunks: 1566
Embeddings: 1566


## **5. FAISS IHNSW**

In [10]:
import faiss

# 🔹 Obtener dimensión del embedding
dimension = embeddings.shape[1]

# 🔹 Crear índice con cosine similarity
index = faiss.IndexFlatIP(dimension)

# 🔹 Agregar embeddings
index.add(embeddings)

print("✅ Nuevo índice FAISS creado")
print("Total vectores indexados:", index.ntotal)

✅ Nuevo índice FAISS creado
Total vectores indexados: 1566


Función de búsqueda semántica

In [11]:
query = "Ratio de Capital Global de CMAC Cusco en el periodo 2015"
q_emb = embedder.encode([query], normalize_embeddings=True)
q_emb = np.array(q_emb).astype("float32")

D, I = index.search(q_emb, 5)

print("🔎 Pregunta:", query)
print("\nResultados encontrados:\n")

for score, idx in zip(D[0], I[0]):
    print("="*60)
    print("Score:", round(float(score), 4))
    print("Entidad:", chunks[idx].metadata["entidad"])
    print("Periodo:", chunks[idx].metadata["periodo"])
    print("\nContenido:\n")
    print(chunks[idx].page_content)
    print("\n")

🔎 Pregunta: Ratio de Capital Global de CMAC Cusco en el periodo 2015

Resultados encontrados:

Score: 0.766
Entidad: CMAC Cusco
Periodo: 2020-12

Contenido:

En el periodo 2020-12, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 382923.16145, por riesgo de mercado de 652.7772 y por riesgo operacional de 59727.09394, sumando un total requerido de 443303.03259. El patrimonio efectivo total fue de 613620.4405700001 y el Ratio de Capital Global alcanzó 13.84.


Score: 0.7612
Entidad: CMAC Cusco
Periodo: 2015-11

Contenido:

En el periodo 2015-11, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 210487.34054, por riesgo de mercado de 459.66722 y por riesgo operacional de 31384.62734, sumando un total requerido de 242331.6351. El patrimonio efectivo total fue de 342837.7145 y el Ratio de Capital Global alcanzó 14.15.


Score: 0.7608
Entidad: CMAC Cusco
Periodo: 2021-02

Contenido:

En el periodo 20

## **6. Retrieval**

In [12]:
def retrieve(query, k=5):
    """Recupera Top-K chunks desde FAISS conservando metadata (source, periodo, entidad)."""
    # 1) Embedding de la query
    query_vector = embedder.encode([query], normalize_embeddings=True).astype("float32")

    # 2) Búsqueda en FAISS
    D, I = index.search(query_vector, k)

    # 3) Construir resultados con metadata
    results = []
    for score, idx in zip(D[0], I[0]):
        ch = chunks[int(idx)]
        results.append({
            "score": float(score),
            "content": ch.page_content,
            "metadata": ch.metadata
        })
    return results


In [13]:
query = "Ratio de Capital Global de CMAC Cusco en 2015-02"

results = retrieve(query, k=3)

print("🔎 Pregunta:", query)
print("\nResultados encontrados:\n")

for r in results:
    print("="*60)
    print("Score:", round(r["score"], 4))
    print(r["content"])
    print("\n")

🔎 Pregunta: Ratio de Capital Global de CMAC Cusco en 2015-02

Resultados encontrados:

Score: 0.7496
En el periodo 2020-12, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 382923.16145, por riesgo de mercado de 652.7772 y por riesgo operacional de 59727.09394, sumando un total requerido de 443303.03259. El patrimonio efectivo total fue de 613620.4405700001 y el Ratio de Capital Global alcanzó 13.84.


Score: 0.7485
En el periodo 2021-02, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 375637.13302, por riesgo de mercado de 2002.91333 y por riesgo operacional de 60393.80375, sumando un total requerido de 438033.85010000004. El patrimonio efectivo total fue de 612886.58843 y el Ratio de Capital Global alcanzó 13.99.


Score: 0.7412
En el periodo 2015-02, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 181729.46033, por riesgo de mercado de 145.51

## **7. Reranker**

> Usamos un modelo de Cross-Encoder que entiende mejor la relación pregunta-respuesta

> FAISS recupera por "cercanía vectorial", el Reranker recupera por "relevancia semántica".

In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def rerank(query, retrieved_docs):
    
    pairs = [[query, doc["content"]] for doc in retrieved_docs]
    
    scores = reranker.predict(pairs)
    
    for doc, score in zip(retrieved_docs, scores):
        doc["rerank_score"] = float(score)
    
    # Ordenar por score del reranker
    reranked = sorted(
        retrieved_docs,
        key=lambda x: x["rerank_score"],
        reverse=True
    )
    
    return reranked

In [16]:
query = "Ratio de Capital Global de CMAC Cusco en 2015-02"

retrieved = retrieve(query, k=5)

reranked = rerank(query, retrieved)

print("🔎 Resultados después de Reranker:\n")

for r in reranked:
    print("="*60)
    print("FAISS score:", round(r["score"], 4))
    print("Rerank score:", round(r["rerank_score"], 4))
    print(r["content"])
    print()

🔎 Resultados después de Reranker:

FAISS score: 0.7412
Rerank score: 8.4541
En el periodo 2015-02, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 181729.46033, por riesgo de mercado de 145.51223000000002 y por riesgo operacional de 27715.13581, sumando un total requerido de 209590.10837. El patrimonio efectivo total fue de 289667.3354 y el Ratio de Capital Global alcanzó 13.82.

FAISS score: 0.737
Rerank score: 7.4477
En el periodo 2015-11, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 210487.34054, por riesgo de mercado de 459.66722 y por riesgo operacional de 31384.62734, sumando un total requerido de 242331.6351. El patrimonio efectivo total fue de 342837.7145 y el Ratio de Capital Global alcanzó 14.15.

FAISS score: 0.7485
Rerank score: 6.2914
En el periodo 2021-02, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 375637.13302, por riesgo

## **8. LLM**

In [17]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2-1.5B-Instruct",
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🔹 Función de generación

In [18]:
def generate_answer(query, top_document):
    
    prompt = f"""
    Usa únicamente la información del siguiente documento para responder.
    Si la información no está presente, responde que no se encontró.

    Documento:
    {top_document}

    Pregunta:
    {query}

    Respuesta:
    """
    
    response = generator(
        prompt,
        max_new_tokens=200,
        temperature=0.2
    )
    
    return response[0]["generated_text"]

🔹 Integración completa

In [19]:
query = "Ratio de Capital Global de CMAC Cusco en 2015-02"

retrieved = retrieve(query, k=5)
reranked = rerank(query, retrieved)

top_doc = reranked[0]["content"]

answer = generate_answer(query, top_doc)

print(answer)

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Usa únicamente la información del siguiente documento para responder.
    Si la información no está presente, responde que no se encontró.

    Documento:
    En el periodo 2015-02, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 181729.46033, por riesgo de mercado de 145.51223000000002 y por riesgo operacional de 27715.13581, sumando un total requerido de 209590.10837. El patrimonio efectivo total fue de 289667.3354 y el Ratio de Capital Global alcanzó 13.82.

    Pregunta:
    Ratio de Capital Global de CMAC Cusco en 2015-02

    Respuesta:
     13.82
"""
# Importing the required libraries
import pandas as pd

def calculate_ratio_of_capital_global(data):
    """
    Calculate the ratio of capital global for a given dataset.

    Parameters:
    data (pandas.DataFrame): A DataFrame containing the required columns.

    Returns:
    float: The calculated ratio of capital global.
    """
    # Extracting the required columns from the 

**8️⃣ LLM – Qwen**

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2-1.5B-Instruct"  # más liviano para evitar errores

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,   # IMPORTANTE
)

model = model.to("cpu")  # Forzar CPU

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

**Construcción del Prompt**

✔ Usar SOLO contexto
✔ Citar por oración
✔ No inventar

In [21]:
def construir_prompt(pregunta, contexto):
    return f"""
Eres un asistente experto en análisis financiero de Cajas Municipales de la SBS.

Usa SOLO la información del contexto.
Si la respuesta no está en el contexto, responde: "No encontrado en los datos".

Cita la fuente en cada oración usando el formato:
[Fuente: nombre_archivo.xlsx]

Contexto:
{contexto}

Pregunta:
{pregunta}

Respuesta:
"""

### **Generación con Qwen**

In [22]:
def generar_respuesta(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.0,      # 🔥 determinístico
        do_sample=False,      # 🔥 SIN sampling
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### **Integrar pipeline RAG**

In [23]:
pregunta = "¿Cuál fue el ratio de Capital Global en 2015-02 para CMAC Cusco?"

# 1) Recuperación FAISS
retrieved = retrieve(pregunta, k=8)

# 2) Reranking
reranked = rerank(pregunta, retrieved)

# 3) Tomar Top-N finales
top_n = reranked[:3]

# 4) Construir contexto (con fuente por bloque para apoyar citación)
contexto = "\n\n".join(
    [f"[Fuente: {d['metadata'].get('source','desconocida')}]\n{d['content']}" for d in top_n]
)

prompt = construir_prompt(pregunta, contexto)

respuesta = generar_respuesta(prompt)

print(respuesta)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Eres un asistente experto en análisis financiero de Cajas Municipales de la SBS.

Usa SOLO la información del contexto.
Si la respuesta no está en el contexto, responde: "No encontrado en los datos".

Cita la fuente en cada oración usando el formato:
[Fuente: nombre_archivo.xlsx]

Contexto:
[Fuente: 2015_Diciembre.xls]
En el periodo 2015-12, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 214665.95494999998, por riesgo de mercado de 480.34396999999996 y por riesgo operacional de 31823.408219999998, sumando un total requerido de 246969.70713999995. El patrimonio efectivo total fue de 346338.99748 y el Ratio de Capital Global alcanzó 14.02.

[Fuente: 2015_Noviembre.xls]
En el periodo 2015-11, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 210487.34054, por riesgo de mercado de 459.66722 y por riesgo operacional de 31384.62734, sumando un total requerido de 242331.6351. El patrimonio efectivo

### **9. Citation per sentence**

Construir contexto correctamente

In [24]:
# Construcción del contexto estructurado
contexto = "\n\n".join(
    [
        f"""
Fuente: {d['metadata']['source']}
Periodo: {d['metadata'].get('periodo','N/A')}
Entidad: {d['metadata'].get('entidad','N/A')}

Contenido:
{d['content']}
"""
        for d in top_n
    ]
)

Prompt fuerte y obligatorio

In [25]:
def construir_prompt(pregunta, contexto):
    return f"""
Eres un asistente financiero especializado en Cajas Municipales de la SBS.

REGLAS OBLIGATORIAS:
1. Usa únicamente información del contexto.
2. Cada oración debe terminar con su fuente.
3. El formato obligatorio es:
   [Fuente: nombre_archivo.xls]
4. No agrupes fuentes.
5. No inventes información.
6. Si el periodo exacto no aparece, responde SOLO:
   No encontrado en los datos.

Contexto:
{contexto}

Pregunta:
{pregunta}

Respuesta:
"""

Generación sin creatividad (clave)

In [26]:
def generar_respuesta(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.0,        # 🔥 sin creatividad
        do_sample=False,        # 🔥 determinístico
        repetition_penalty=1.2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Validación

In [27]:
import re

def validar_citacion(respuesta):
    oraciones = [o.strip() for o in respuesta.split(".") if o.strip()]
    for o in oraciones:
        if "[Fuente:" not in o:
            return False
    return True

if validar_citacion(respuesta):
    print("✅ Citation per sentence correcto")
else:
    print("⚠ Falta citación por oración")

⚠ Falta citación por oración


In [28]:
def consultar_sbs(pregunta, k=10, top_n=3):
    
    # 🔹 1. Recuperación FAISS
    retrieved = retrieve(pregunta, k=k)
    
    if len(retrieved) == 0:
        return "No encontrado en los datos."
    
    # 🔹 2. Reranking
    reranked = rerank(pregunta, retrieved)
    
    # 🔹 3. Seleccionar Top-N
    top_docs = reranked[:top_n]
    
    # 🔹 4. Construcción de contexto estructurado
    contexto = "\n\n".join(
        [
            f"""
Fuente: {d['metadata']['source']}

Contenido:
{d['content']}
"""
            for d in top_docs
        ]
    )
    
    # 🔹 5. Construcción de prompt
    prompt = construir_prompt(pregunta, contexto)
    
    # 🔹 6. Generación LLM
    respuesta = generar_respuesta(prompt)
    
    return respuesta

In [29]:
pregunta = "¿Cuál fue el Ratio de Capital Global en 2015-12 para CMAC Cusco?"

respuesta = consultar_sbs(pregunta)

print(respuesta)


Eres un asistente financiero especializado en Cajas Municipales de la SBS.

REGLAS OBLIGATORIAS:
1. Usa únicamente información del contexto.
2. Cada oración debe terminar con su fuente.
3. El formato obligatorio es:
   [Fuente: nombre_archivo.xls]
4. No agrupes fuentes.
5. No inventes información.
6. Si el periodo exacto no aparece, responde SOLO:
   No encontrado en los datos.

Contexto:

Fuente: 2015_Diciembre.xls

Contenido:
En el periodo 2015-12, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 214665.95494999998, por riesgo de mercado de 480.34396999999996 y por riesgo operacional de 31823.408219999998, sumando un total requerido de 246969.70713999995. El patrimonio efectivo total fue de 346338.99748 y el Ratio de Capital Global alcanzó 14.02.



Fuente: 2015_Noviembre.xls

Contenido:
En el periodo 2015-11, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 210487.34054, por riesgo de merc

### **10.  Hallucination Guard**

Función de verificación

In [30]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def hallucination_guard(answer, source_document):
    
    result = {}
    
    # -------------------------------------------------
    # 1️⃣ Verificación exacta de contenido
    # -------------------------------------------------
    exact_match = answer.strip() in source_document
    
    # -------------------------------------------------
    # 2️⃣ Extraer números de la respuesta
    # -------------------------------------------------
    numbers_in_answer = re.findall(r"\d+\.\d+|\d+", answer)
    
    missing_numbers = []
    
    for num in numbers_in_answer:
        if num not in source_document:
            missing_numbers.append(num)
    
    # -------------------------------------------------
    # 3️⃣ Similaridad semántica
    # -------------------------------------------------
    vectorizer = TfidfVectorizer().fit([answer, source_document])
    vectors = vectorizer.transform([answer, source_document])
    similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
    
    # -------------------------------------------------
    # 4️⃣ Evaluación final
    # -------------------------------------------------
    
    if exact_match:
        status = "✅ Totalmente fundamentada"
    elif similarity > 0.7 and len(missing_numbers) == 0:
        status = "✅ Fundamentada (alta similitud)"
    else:
        status = "⚠ Posible alucinación"
    
    result["status"] = status
    result["semantic_similarity"] = round(similarity, 4)
    result["missing_numbers"] = missing_numbers
    
    return result

Uso

In [31]:
guard_result = hallucination_guard(answer, top_doc)

print("Estado:", guard_result["status"])
print("Similitud:", guard_result["semantic_similarity"])
print("Números faltantes:", guard_result["missing_numbers"])

Estado: ⚠ Posible alucinación
Similitud: 0.6144
Números faltantes: []


1 Similitud = 0.7635

Eso es alto.
Significa que la respuesta está bastante alineada con el documento.

No es alucinación fuerte.


2¿Por qué apareció 100?

Probablemente el LLM escribió algo como:

El Ratio fue 13.82%, equivalente a 13.82 por cada 100 unidades de capital.

Ese “100” no está en el texto original.

**11.Grounding evaluation**

In [32]:
    ####IMPLEMENTACION
import re


def grounding_score(answer, source_document):
    
    # Limpiar texto
    answer_tokens = re.findall(r"\w+", answer.lower())
    source_tokens = set(re.findall(r"\w+", source_document.lower()))
    
    # Contar tokens respaldados
    supported_tokens = [t for t in answer_tokens if t in source_tokens]
    
    if len(answer_tokens) == 0:
        return 0
    
    score = len(supported_tokens) / len(answer_tokens)
    
    return round(score, 4)

In [33]:
score = grounding_score(answer, top_doc)

print("Grounding score:", score)

Grounding score: 0.543


Aproximadamente el 50% de los tokens de la respuesta aparecen literalmente en el documento.

###**12.BLEU/ROUGE**

1️⃣ BLEU

In [34]:
reference_answer = "En el periodo 2015-02, el Ratio de Capital Global de CMAC Cusco fue 13.82%."
generated_answer = answer

In [35]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_bleu(reference, generated):
    
    ref_tokens = reference.lower().split()
    gen_tokens = generated.lower().split()
    
    smoothie = SmoothingFunction().method1
    
    score = sentence_bleu(
        [ref_tokens],
        gen_tokens,
        smoothing_function=smoothie
    )
    
    return round(score, 4)

In [36]:
reference_answer = "En el periodo 2015-02, el Ratio de Capital Global de CMAC Cusco fue 13.82%."
generated_answer = answer

bleu_score = compute_bleu(reference_answer, generated_answer)

print("BLEU score:", bleu_score)

BLEU score: 0.0495


BLEU penaliza muchísimo cuando:

Cambia orden de palabras

Cambia conectores

Cambia “fue” por “alcanzó”

Agrega contexto adicional

Reestructura la frase

Ejemplo:

Referencia:

El Ratio fue 13.82%.

Generado:

El Ratio de Capital Global alcanzó 13.82%.

Semánticamente correcto
BLEU puede bajar a 0.1–0.3

**Implementación rápida ROUGE**

In [37]:
from rouge_score import rouge_scorer

def compute_rouge(reference, generated):
    
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=True
    )
    
    scores = scorer.score(reference, generated)
    
    return {
        "ROUGE-1": round(scores['rouge1'].fmeasure, 4),
        "ROUGE-L": round(scores['rougeL'].fmeasure, 4)
    }

rouge_scores = compute_rouge(reference_answer, generated_answer)

print("ROUGE scores:", rouge_scores)

ROUGE scores: {'ROUGE-1': 0.1429, 'ROUGE-L': 0.1339}


ROUGE mide superposición de palabras exactas.

Si tu respuesta generada:

Reformuló

Resumió

Cambió estructura

Quitó partes largas del documento

Entonces ROUGE baja.

Ejemplo típico:

Documento:

En el periodo 2015-02, la entidad CMAC Cusco presentó un requerimiento de patrimonio efectivo por riesgo de crédito de 181729.46033...

LLM responde:

En 2015-02, el Ratio de Capital Global de CMAC Cusco fue 13.82%.

La respuesta es correcta, pero mucho más corta.

ROUGE penaliza porque no repite todo el texto.